# Pandas — Technical Reference

## Quick Index

| Pattern | When to use | Section |
| :--- | :--- | :--- |
| Data inspection | First look at any DataFrame | §1 |
| Selection & filtering | Subset rows and columns | §2 |
| NULL handling | Missing values | §3 |
| Data manipulation | Clean, transform, create columns | §4 |
| Joins & merges | Combine DataFrames | §5 |
| Aggregation & GroupBy | Summarize, reshape, pivot | §6 |
| Window operations | Rank, shift, cumulative, rolling | §7 |
| Date & time | Date arithmetic, extraction | §8 |
| String operations | Text cleaning and extraction | §9 |
| I/O | Read and write data | §10 |
| Performance tips | Speed and memory optimization | §11 |

---
## When to Use

| Signal | Pandas method to reach for |
| :--- | :--- |
| First look at a DataFrame | `.info()`, `.describe()`, `.head()` |
| Filter rows by condition | `.query()` or boolean indexing |
| Select rows and columns by label | `.loc[rows, cols]` |
| Select rows and columns by position | `.iloc[rows, cols]` |
| Replace NULLs | `.fillna()` or `.ffill()` / `.bfill()` |
| Apply condition to create a column | `np.where()` (binary) or `np.select()` (multi) |
| Summarize with aggregation | `.groupby().agg()` |
| Apply aggregation but keep original shape | `.groupby().transform()` |
| Rank within a group | `.groupby().rank(method='dense')` |
| Compare to previous / next row | `.groupby().shift(1)` / `.shift(-1)` |
| Running total within a group | `.groupby().cumsum()` |
| Pivot rows to columns | `.pivot_table()` |
| Combine two DataFrames on a key | `.merge()` |
| Find rows with no match | `merge(..., how='left', indicator=True)` |
| Moving average | `.rolling(n).mean()` |
| Parse date strings | `pd.to_datetime()` |
| Extract date parts | `.dt.year`, `.dt.month`, `.dt.dayofweek` |
| String pattern match | `.str.contains()` |
| Extract regex groups from string | `.str.extract()` |

---
## §1 — Data Inspection

Always run these first. They tell you shape, dtypes, missing values, and distributions before you touch the data.

In [ ]:
import pandas as pd
import numpy as np

# Shape and preview
df.shape                        # (rows, cols)
df.head(5)                      # first 5 rows
df.tail(5)                      # last 5 rows

# Column types and null counts
df.info()                       # dtypes + non-null counts per column
df.dtypes                       # just the dtypes

# Summary statistics
df.describe()                   # numeric columns: count, mean, std, min, quartiles, max
df.describe(include='all')      # include categorical columns too

# Cardinality and distributions
df['col'].nunique()             # number of distinct values
df['col'].value_counts()        # frequency of each value, sorted descending
df['col'].value_counts(normalize=True)  # as proportions (0–1)

# NULL summary across all columns
df.isna().sum()                 # null count per column
df.isna().sum() / len(df)       # null rate per column

# ── COMPARE — row-level diff of two like-shaped frames ───────────────────────
before.compare(after)                             # only changed cells, self/other
before.compare(after, keep_equal=True)            # show unchanged values too
before.compare(after, align_axis=0)               # stack instead of side-by-side


**Common mistakes:**
- Skipping `.info()` and assuming dtypes — date columns often load as `object` and need explicit parsing
- Using `.describe()` without checking `.info()` first — `describe()` silently skips non-numeric columns unless `include='all'` is set
- Confusing `df['col'].count()` (excludes NaN) with `len(df)` (includes all rows)

`.compare()` sits here rather than in §5 because it is a diff, not a join: it takes two identically-labelled frames and returns only the cells that differ.


---
## §2 — Selection & Filtering

Three distinct tools: `[]` for quick column access, `.loc` for label-based selection, `.iloc` for position-based selection. Always use `.loc` when selecting rows and columns together.

column selection by pattern, fast scalar access, locating the *row* of an extreme value, and MultiIndex slicing.


In [ ]:
# Column selection
df['col']                           # single column → Series
df[['col1', 'col2']]                # multiple columns → DataFrame

# Select columns by name pattern (no boolean mask needed)
df.filter(like='revenue')                 # columns containing 'revenue'
df.filter(regex=r'_(usd|eur)$')           # columns matching a regex
df.filter(items=['a', 'b'])               # explicit list, silently skips missing
df.filter(like='2024', axis=0)            # can filter the INDEX instead

# .loc — label-based (rows by index label, cols by name)
df.loc[0]                           # row with index label 0
df.loc[0:5, 'col1':'col3']          # rows 0–5, columns col1 through col3 (inclusive)
df.loc[mask, ['col1', 'col2']]      # filtered rows, specific columns

# .iloc — position-based (integer index only)
df.iloc[0]                          # first row
df.iloc[0:5, 0:3]                   # first 5 rows, first 3 columns (exclusive end)
df.iloc[-1]                         # last row

# .at / .iat — scalar access only, but much faster than .loc / .iloc
df.at[3, 'price']                         # label-based single value
df.iat[3, 2]                              # position-based single value
df.at[3, 'price'] = 9.99                  # assignment works too

# Boolean indexing
df[df['age'] > 30]                  # single condition
df[(df['age'] > 30) & (df['region'] == 'US')]   # AND — use & not 'and'
df[(df['age'] < 18) | (df['age'] > 65)]         # OR  — use | not 'or'
df[~(df['status'] == 'inactive')]               # NOT — use ~
df[df['platform'].isin(['iOS', 'Android'])]     # IN
df[~df['user_id'].isin(blocked_ids)]            # NOT IN

# .query() — readable string syntax
df.query("age > 30 and region == 'US'")
df.query("`column name` > 0")                   # backticks for column names with spaces
threshold = 100
df.query("revenue > @threshold")                # @ prefix to reference Python variables

# Locate the ROW of a max/min, not the value
df['price'].idxmax()                      # index label of the largest price
df.loc[df['price'].idxmax()]              # the whole winning row
df.loc[df.groupby('cat')['price'].idxmax()]   # best row PER category

# MultiIndex selection
mi = df.set_index(['region', 'date'])
mi.loc['West']                            # all rows for one outer level
mi.loc[('West', '2024-01-01')]            # a specific (outer, inner) pair
mi.xs('2024-01-01', level='date')         # slice on an INNER level
mi.xs('West', level='region', drop_level=False)   # keep the level in the result

idx = pd.IndexSlice
mi.loc[idx['West':'East', '2024-01':'2024-03'], :]   # range on both levels

# ── INDEX SET OPERATIONS — often clearer than a merge ────────────────────────
a.index.intersection(b.index)             # in both
a.index.union(b.index)                    # in either
a.index.difference(b.index)               # in a only
a.index.symmetric_difference(b.index)     # in exactly one
df.loc[a.index.difference(b.index)]       # rows dropped between two snapshots


**`.loc` vs `.iloc` vs `[]` comparison:**

| | Rows | Columns | End inclusive? |
| :--- | :--- | :--- | :--- |
| `[]` | ❌ (only with boolean mask) | ✅ by name | n/a |
| `.loc` | ✅ by label | ✅ by name | ✅ Yes |
| `.iloc` | ✅ by position | ✅ by position | ❌ No |

**`.loc` vs `.at` for a single value** (timings measured in `Reference_Performance` §3)**:**

| | Accepts | Returns | Relative speed |
| :--- | :--- | :--- | :--- |
| `.loc[r, c]` | labels, slices, masks, lists | scalar *or* Series/DataFrame | 1× |
| `.at[r, c]` | a single label pair only | always a scalar | ~1.3–1.5× faster |

**Common mistakes:**
- Using `and` / `or` / `not` instead of `&` / `|` / `~` in boolean indexing — raises `ValueError`
- Forgetting parentheses around each condition: `df[df['a'] > 1 & df['b'] == 2]` evaluates incorrectly; always wrap each condition
- Chained indexing `df['col'][mask]` instead of `df.loc[mask, 'col']` — triggers `SettingWithCopyWarning` and may not modify the original
- Using `.max()` when you want `.idxmax()` — `.max()` gives the value and loses which row produced it
- `.idxmax()` on an all-NaN column raises; guard with `.notna().any()` first
- Calling `.at` with a slice or list — it only accepts single labels, unlike `.loc`
- Forgetting `pd.IndexSlice` and writing `mi.loc['West':'East', '2024-01':]` directly, which is ambiguous across levels

Index set operations (`union` / `intersection` / `difference` / `symmetric_difference`) are selection tools rather than joins — use them to ask which labels two frames share, or which rows were added or dropped between two snapshots.


---
## §3 — NULL Handling

NULLs affect filtering, aggregation, and joins. Understand the behavior before writing any pipeline.

`np.nan` is a float, which is why an integer column with one missing value silently becomes `float64`. The nullable dtypes fix that.


In [ ]:
# Detect NULLs
df.isna()                                       # IS NULL — element-wise boolean
df.notna()                                      # IS NOT NULL
df['col'].isna().sum()                          # count NULLs in a column

# Fill NULLs
df['salary'].fillna(0)                          # IFNULL(salary, 0)
df['contact'] = df['phone'].fillna(df['email']).fillna('Unknown')  # COALESCE(phone, email, 'Unknown')
df['col'].fillna(df['col'].mean())              # fill with column mean

# Different fill value per column, in one call
df.fillna({'salary': 0, 'city': 'Unknown', 'score': df['score'].median()})

# Forward / backward fill within a group
df['value'] = df.groupby('user_id')['value'].ffill()  # fill from previous row in group
df['value'] = df.groupby('user_id')['value'].bfill()  # fill from next row in group

# Limit how far a fill propagates
df['v'].ffill(limit=2)                    # carry forward at most 2 rows

# Drop rows with NULLs
df.dropna()                                     # drop rows with ANY null
df.dropna(subset=['user_id', 'date'])           # drop rows where specific columns are null
df.dropna(how='all')                            # drop rows where ALL values are null

# NULL behavior in aggregations
df['col'].sum()     # NaN ignored (treated as 0 in sum)
df['col'].mean()    # NaN ignored — denominator is non-null count, not len(df)
df['col'].count()   # excludes NaN
len(df)             # includes NaN rows

# ── np.nan vs pd.NA vs pd.NaT ────────────────────────────────────────────────
# np.nan  float missing        -> forces the column to float64
# pd.NaT  datetime missing
# pd.NA   dtype-agnostic missing, used by the nullable extension dtypes

s = pd.Series([1, 2, None])
s.dtype                                   # float64  <- int column silently widened

s = pd.Series([1, 2, None], dtype='Int64')    # capital I: nullable integer
s.dtype                                   # Int64, values stay integers, missing is pd.NA

# Nullable dtypes across the board
df = df.convert_dtypes()                  # infer Int64 / string / boolean automatically
df['flag'].astype('boolean')              # nullable bool: True / False / pd.NA
df['name'].astype('string')               # nullable string (not object)

# Three-valued logic: comparisons with pd.NA propagate rather than returning False
(pd.NA > 1)                               # <NA>, not False
pd.Series([True, pd.NA], dtype='boolean').any()    # True  (NA ignored)
pd.Series([False, pd.NA], dtype='boolean').all()   # False


**Missing-value markers:**

| Marker | Belongs to | Forces float? | Equal to itself? |
| :--- | :--- | :--- | :--- |
| `np.nan` | numpy float | ✅ yes | ❌ no |
| `pd.NaT` | datetime / timedelta | n/a | ❌ no |
| `pd.NA` | nullable extension dtypes | ❌ no | ❌ no (propagates) |

**Common mistakes:**
- Using `df['col'] == None` or `df['col'] == np.nan` — NaN is never equal to anything, including itself; always use `.isna()`
- `fillna(0)` before `.mean()` when NULLs should be excluded — changes the denominator and deflates the mean
- `ffill()` without sorting first — fills in row order, not logical time order; always `sort_values` before forward filling
- Expecting `Int64` (nullable) to behave like `int64` (numpy) — only the capitalised one accepts missing values
- Filtering with `df[df['flag']]` on a `boolean` column containing `pd.NA` — raises, because NA is not a valid mask value; use `df[df['flag'].fillna(False)]`
- Assuming `convert_dtypes()` is free — it copies the frame and can be slow on wide data


---
## §4 — Data Manipulation

Covers column creation, renaming, type casting, and conditional logic — the day-to-day cleaning toolkit.

`.pipe()` completes the chainable trio with `.assign()` and `.query()`; the rest are the MultiIndex/alignment tools that show up the moment a groupby returns more than one level.


In [ ]:
# Add / overwrite columns
df['new_col'] = df['a'] + df['b']              # vectorized arithmetic
df = df.assign(new_col=df['a'] + df['b'])      # chainable version

# .pipe — insert your own function into a method chain
def add_margin(d, rate):
    return d.assign(margin=d['revenue'] * rate)

(df
   .query('revenue > 0')
   .pipe(add_margin, rate=0.3)            # instead of add_margin(df.query(...), 0.3)
   .sort_values('margin', ascending=False)
   .head(10))

# Rename columns
df.rename(columns={'old': 'new', 'a': 'b'})   # rename specific columns
df.columns = ['col1', 'col2', 'col3']          # rename all columns at once

# Drop columns or rows
df.drop(columns=['col1', 'col2'])              # drop columns
df.drop(index=[0, 1, 2])                       # drop rows by index label

# Type casting
df['col'].astype(int)
df['col'].astype(float)
df['col'].astype(str)
df['col'].astype('category')                   # memory-efficient for low-cardinality strings

# Conditional logic
# np.where — binary (IF/ELSE)
df['spender_type'] = np.where(df['spend'] > 100, 'High', 'Low')

# np.select — multi-condition (CASE WHEN equivalent)
conditions = [
    df['spend'] > 100,
    df['spend'].between(50, 100)
]
choices = ['High', 'Medium']
df['spender_type'] = np.select(conditions, choices, default='Low')

# pd.cut — bin continuous values into labeled ranges
df['age_group'] = pd.cut(
    df['age'],
    bins=[0, 18, 35, 60, 100],
    labels=['Under 18', '18-35', '36-60', '60+']
)

# map — replace values using a dictionary (SQL CASE WHEN on discrete values)
df['platform_label'] = df['platform_code'].map({1: 'iOS', 2: 'Android', 3: 'Web'})

# apply — row-wise or column-wise custom function (use sparingly — slow)
df['col'] = df['col'].apply(lambda x: x.strip().lower())  # acceptable on strings
df['result'] = df.apply(lambda row: row['a'] + row['b'], axis=1)  # avoid if vectorized op exists

# Deduplication
df.drop_duplicates()                           # drop fully duplicate rows
df.drop_duplicates(subset=['user_id'])         # keep first occurrence per user_id
df.drop_duplicates(subset=['user_id'], keep='last')  # keep last occurrence

# sort_values(key=) — sort by a transform WITHOUT adding a helper column
df.sort_values('name', key=lambda s: s.str.lower())        # case-insensitive
df.sort_values('code', key=lambda s: s.str.len())          # by string length
df.sort_values(['dept', 'sal'], ascending=[True, False])   # mixed directions

# MultiIndex plumbing
g = df.groupby(['region', 'year'])['sales'].agg(['sum', 'mean'])
g.droplevel(0)                            # drop an index level
g.droplevel(0, axis=1)                    # drop a COLUMN level (after multi-agg)
g.swaplevel(0, 1).sort_index()            # reorder levels, then re-sort
g.columns = ['_'.join(c) for c in g.columns]   # flatten a MultiIndex column

# align — put two objects on a common index before combining
a2, b2 = a.align(b, join='outer', axis=0, fill_value=0)
a2 + b2                                   # now safe: identical index


**Choosing a value-substitution / labeling tool:**

| | Selects on | Unmatched values become | SQL analogue |
| :--- | :--- | :--- | :--- |
| `np.where(cond, x, y)` | one boolean condition | n/a (binary) | `IF` / `CASE WHEN ... ELSE` |
| `np.select(conds, choices, default)` | many conditions (first wins) | `default` | multi-branch `CASE WHEN` |
| `s.map(dict)` | discrete value lookup | **NaN** | `CASE` on discrete values |
| `s.replace(dict)` | discrete value lookup | **unchanged** | targeted `REPLACE` |

The `map`-vs-`replace` split is the gotcha: `map` blanks out keys it doesn't know, `replace` leaves them alone.

**`where` vs `mask` (exact opposites):**

| | Keeps original value where | Replaces where |
| :--- | :--- | :--- |
| `s.where(cond, other)` | cond is **True** | cond is False |
| `s.mask(cond, other)` | cond is **False** | cond is True |

**`map` vs `apply` (on a Series):** use `map` for a dict/Series lookup or simple function (faster, clearer); use `apply` only for arbitrary functions a `map` can't express.

**`pd.cut` vs `pd.qcut`:** `cut` makes equal-**width** bins from edges you supply; `qcut` makes equal-**frequency** (quantile) bins. Use `qcut` for balanced bucket sizes, `cut` when the thresholds carry meaning.




**Common mistakes:**
- Conditions in `np.select` are evaluated top to bottom — first matching condition wins; order matters
- `df['col'].map(dict)` returns NaN for keys not in the dictionary — add `.fillna()` if needed
- Using `apply(axis=1)` (row-wise) when a vectorized operation exists — it's a Python loop and 10–100x slower
- `inplace=True` does not speed things up and makes chaining impossible — avoid it
- Adding a temporary column just to sort by it, then dropping it — `sort_values(key=)` does it in one step
- Forgetting that `key=` receives the **whole Series**, not one element, so `key=lambda x: x.lower()` fails; use `x.str.lower()`
- Arithmetic on two Series with different indexes producing surprise NaNs — pandas aligns on index first; use `.align()` or `.reset_index(drop=True)` deliberately
- Leaving a MultiIndex on columns after `.agg(['sum','mean'])` and then failing to select by name


In [2]:
import pandas as pd
s = pd.Series([10, 20, 30, 40])
print("where(s>20) keeps where TRUE :", s.where(s>20, 0).tolist())
print("mask(s>20)  keeps where FALSE:", s.mask(s>20, 0).tolist())

vals = pd.Series([1,2,3,4,5,6,7,8,100])
print("\ncut  (equal-width) bucket sizes:", pd.cut(vals, bins=3).value_counts().sort_index().tolist())
print("qcut (equal-freq)  bucket sizes:", pd.qcut(vals, q=3).value_counts().sort_index().tolist())


where(s>20) keeps where TRUE : [0, 0, 30, 40]
mask(s>20)  keeps where FALSE: [10, 20, 0, 0]

cut  (equal-width) bucket sizes: [8, 0, 1]
qcut (equal-freq)  bucket sizes: [3, 3, 3]


---
## §5 — Joins & Merges

`.merge()` is the primary tool — use it for all column-based joins. `.join()` joins on index and is rarely needed.

`cross` builds every combination, `merge_asof` joins on *nearest earlier* key rather than equality, and `.compare()` answers "what changed between these two frames?".


In [ ]:
# Basic merge — SQL JOIN equivalents
A.merge(B, on='id', how='inner')        # INNER JOIN
A.merge(B, on='id', how='left')         # LEFT JOIN
A.merge(B, on='id', how='right')        # RIGHT JOIN
A.merge(B, on='id', how='outer')        # FULL OUTER JOIN
A.merge(B, how='cross')                 # CROSS JOIN — no key, every combination
# cross returns len(A) * len(B) rows: 10k x 10k is 100M. Filter first.

# Scaffold pattern — the main reason to reach for cross.
# Build every (user, day) combination so "no activity" is a zero, not a gap.
scaffold = users[['user_id']].merge(
    pd.DataFrame({'date': pd.date_range('2024-01-01', '2024-01-31')}),
    how='cross'
)
# then left-join actual events onto the scaffold -> zero-filled gaps
full = scaffold.merge(events, on=['user_id', 'date'], how='left').fillna({'n': 0})

# Other cross uses: parameter grids, all pairwise combinations
params = grid_a.merge(grid_b, how='cross')

# Pre-1.2 equivalent, still seen in older code:
left.assign(_k=1).merge(right.assign(_k=1), on='_k').drop(columns='_k')

# Join on multiple keys
A.merge(B, on=['user_id', 'date'], how='left')

# Join on differently named columns
A.merge(B, left_on='user_id', right_on='id', how='inner')

# Handle overlapping column names
A.merge(B, on='id', how='left', suffixes=('_a', '_b'))

# ── MERGE ON INDEX ───────────────────────────────────────────────────────────
left.merge(right, left_index=True, right_index=True, how='inner')
left.merge(right, left_on='user_id', right_index=True, how='left')   # mixed
left.join(right, how='left')              # .join() defaults to index-on-index

# Find rows in A with NO match in B — LEFT JOIN ... WHERE B.id IS NULL
merged = A.merge(B, on='id', how='left', indicator=True)
no_match = merged.loc[merged['_merge'] == 'left_only'].drop(columns='_merge')

# Self join — join a DataFrame to itself
employees.merge(
    employees[['id', 'name']].rename(columns={'id': 'manager_id', 'name': 'manager_name'}),
    on='manager_id',
    how='left'
)

# ── MERGE_ASOF — join to the most recent EARLIER row ─────────────────────────
# Both frames MUST be sorted on the `on` key first.
trades = trades.sort_values('time')
quotes = quotes.sort_values('time')

pd.merge_asof(trades, quotes, on='time')                    # nearest earlier quote
pd.merge_asof(trades, quotes, on='time', by='symbol')       # match within symbol
pd.merge_asof(trades, quotes, on='time',
              tolerance=pd.Timedelta('2min'),               # else leave NaN
              direction='backward')                         # 'forward' | 'nearest'

# ── MERGE_ORDERED — outer merge preserving order, with optional group fill ───
pd.merge_ordered(df1, df2, on='date', fill_method='ffill', left_by='group')

# Comparing two frames rather than joining them? .compare() is in §1;
# set logic on indexes (union / intersection / difference) is in §2.

# Stack DataFrames vertically — SQL UNION ALL
pd.concat([df1, df2], ignore_index=True)        # UNION ALL
pd.concat([df1, df2]).drop_duplicates()         # UNION (distinct)


**`merge` vs `join`, and `concat` vs `merge`:**

| | Aligns on | Default keys | Prefer when |
| :--- | :--- | :--- | :--- |
| `A.merge(B, on='id')` | **columns** | none — you specify `on` | Almost always — explicit, SQL-like, handles renamed keys |
| `A.join(B)` | the **index** | the index | You've set a meaningful index and want to glue several frames on it |

`.join()` silently aligns by index/position, so if your key is an ordinary column it matches the **wrong rows**. Prefer `.merge()` unless you deliberately set an index.

| | Combines by | Use for |
| :--- | :--- | :--- |
| `pd.concat([A, B])` | stacking rows (or columns) — no key matching | `UNION ALL` / gluing partitions |
| `A.merge(B, on=k)` | matching a key | `JOIN` |


**Choosing a join:**

| Goal | Tool |
| :--- | :--- |
| Match on equal keys | `merge(on=...)` |
| Every combination, no key | `merge(how='cross')` |
| Match on *nearest earlier* key | `merge_asof(direction='backward')` |
| Match on nearest key either side | `merge_asof(direction='nearest')` |
| Align on index | `merge(left_index=True, right_index=True)` or `.join()` |
| Stack rows / columns | `pd.concat` |
| Fill gaps from a second frame | `combine_first` |
| See what changed | `.compare()` — code in §1 |
| Which keys are shared | `Index.intersection` / `.difference` — code in §2 |

**Common mistakes:**
- Merging on columns with different dtypes (e.g. `int` vs `str`) — merge silently produces no matches; cast first
- Forgetting `suffixes` when both DataFrames have overlapping column names — pandas adds `_x` / `_y` by default
- Using `.join()` instead of `.merge()` — `.join()` joins on index by default; `.merge()` is almost always what you want
- `pd.concat` without `ignore_index=True` — preserves original indices, causing duplicate index values
- Running `how='cross'` on two large frames — output is the **product** of row counts; 10k × 10k is 100M rows. Filter first
- Calling `merge_asof` on unsorted input — it does not raise, it returns wrong matches. Sort both sides on the `on` key
- Using `merge_asof(..., by=...)` and forgetting the `by` column must also be present in both frames
- Omitting `tolerance` in `merge_asof`, which happily matches a quote from three weeks earlier
- Expecting `.compare()` to work on differently-shaped frames — it requires identical labels; reindex first


In [1]:
import pandas as pd
A = pd.DataFrame({"id":[1,2,3], "val":["a","b","c"]})
B = pd.DataFrame({"id":[2,3,4], "score":[20,30,40]})

print("merge on 'id' (inner) — matches on the key:")
print(A.merge(B, on="id", how="inner"))

print("\njoin — aligns on INDEX, so it pairs the WRONG rows:")
print(A.join(B, lsuffix="_A", rsuffix="_B"))


merge on 'id' (inner) — matches on the key:
   id val  score
0   2   b     20
1   3   c     30

join — aligns on INDEX, so it pairs the WRONG rows:
   id_A val  id_B  score
0     1   a     2     20
1     2   b     3     30
2     3   c     4     40


---
## §6 — Aggregation & GroupBy

Three distinct tools with different output shapes: `agg` collapses rows, `transform` preserves shape, `pivot_table` reshapes across two dimensions.

returning the winning *row* per group, grouping by an index level, and grouping by time and category together.


In [ ]:
# SQL pipeline equivalent
# SELECT col1, AGG(col2) FROM t WHERE cond GROUP BY col1 HAVING ... ORDER BY ... LIMIT n
result = (
    df.query('condition')                           # WHERE
      .groupby('col1', as_index=False)              # GROUP BY
      .agg(agg_col=('col2', 'sum'))                 # SELECT + aggregate
      .query('agg_col > 100')                       # HAVING
      .sort_values('col1', ascending=False)         # ORDER BY
      .head(10)                                     # LIMIT
)

# Basic aggregations
len(df)                                 # COUNT(*)
df['col'].count()                       # COUNT(col) — excludes NULLs
df['user_id'].nunique()                 # COUNT(DISTINCT user_id)
df['sales'].sum()                       # SUM
df['price'].mean()                      # AVG

# Named aggregation in groupby — preferred syntax
result = df.groupby('region', as_index=False).agg(
    total_sales  = ('sales',   'sum'),
    avg_price    = ('price',   'mean'),
    unique_users = ('user_id', 'nunique'),
    min_score    = ('score',   'min')
)

# Conditional aggregation — SUM(CASE WHEN ...) equivalent
df['ios_rev'] = df['revenue'].where(df['platform'] == 'iOS', 0)        # else 0
df['ios_rev_excl'] = df['revenue'].where(df['platform'] == 'iOS')      # else NaN (excluded from mean)
result = df.groupby('region', as_index=False).agg(
    ios_total = ('ios_rev',      'sum'),
    ios_avg   = ('ios_rev_excl', 'mean')
)

# idxmax / idxmin inside a groupby — the row, not just the value
best = df.loc[df.groupby('category')['revenue'].idxmax()]      # top row per category
df.groupby('category')['revenue'].idxmin()                     # index labels only

# Equivalent alternatives, and when each wins
df.sort_values('revenue').groupby('category').tail(1)          # ties -> keeps one
df.loc[df.groupby('category')['revenue'].transform('max') == df['revenue']]  # keeps ALL ties

# transform — aggregation that keeps original shape (window function equivalent)
df['group_total'] = df.groupby('region')['revenue'].transform('sum')
df['pct_of_group'] = df['revenue'] / df['group_total']

# pivot_table — GROUP BY across two categorical dimensions
df.pivot_table(
    index='region',                     # rows
    columns='platform',                 # columns
    values='revenue',                   # values
    aggfunc='sum',                      # aggregation
    fill_value=0                        # replace NaN with 0
).reset_index()

# crosstab — frequency count only (pivot_table shortcut)
pd.crosstab(df['region'], df['platform'])

# Group by an INDEX level rather than a column
mi = df.set_index(['region', 'date'])
mi.groupby(level='region')['sales'].sum()
mi.groupby(level=[0, 1]).size()
mi.groupby(level='region', group_keys=False).apply(lambda g: g.head(2))

# pd.Grouper — time buckets combined with ordinary columns
df.groupby(pd.Grouper(key='order_date', freq='ME'))['amount'].sum()   # month end
df.groupby([pd.Grouper(key='order_date', freq='W'), 'region'])['amount'].sum()
df.set_index('order_date').groupby([pd.Grouper(freq='QE'), 'channel']).size()


**`agg` vs `transform` vs `pivot_table`:**

| | Output shape | SQL equivalent | Use when |
| :--- | :--- | :--- | :--- |
| `agg` | Collapsed (one row per group) | `GROUP BY` | You want a summary table |
| `transform` | Same as input | Window function | You need the result back on every row |
| `pivot_table` | Reshaped (groups become columns) | `GROUP BY` + conditional agg | Two categorical dimensions |

**Getting the top row per group:**

| Approach | Ties | Speed | Note |
| :--- | :--- | :--- | :--- |
| `loc[groupby.idxmax()]` | keeps one arbitrarily | fast | fails on all-NaN groups |
| `sort_values().groupby().tail(1)` | keeps one | medium | order is explicit |
| `transform('max') == col` | keeps **all** ties | medium | usually the correct answer |
| `groupby().apply(nlargest)` | configurable | slowest | avoid on large frames |

**Common mistakes:**
- Forgetting `as_index=False` in `groupby` — group keys become the index, not columns
- Using `agg` when you need `transform` — `agg` collapses rows; you lose the original index
- `pivot_table` without `fill_value=0` — missing combinations produce NaN instead of 0
- `.agg(['sum', 'mean'])` with a list produces a MultiIndex — use named aggregation syntax to avoid this
- Using `.idxmax()` on a group that is entirely NaN — raises `ValueError`; filter empty groups first
- Assuming `idxmax` breaks ties fairly; it returns the first occurrence
- `pd.Grouper(freq='M')` is deprecated in favour of `'ME'` (month **end**) in pandas 2.2+; `'MS'` is month start
- Grouping by a datetime *column* without `pd.Grouper` — you get one group per distinct timestamp


**`pivot` vs `pivot_table`, and three "count" methods:**

| | Aggregates duplicates? | Takes `aggfunc`? | Errors on dup keys? |
| :--- | :--- | :--- | :--- |
| `df.pivot(...)` | ❌ reshape only | ❌ | ✅ raises `ValueError` |
| `df.pivot_table(...)` | ✅ | ✅ | ❌ aggregates them |

Use `pivot` only when each index/column pair is already unique; otherwise `pivot_table`.

| Method | Counts | SQL equivalent |
| :--- | :--- | :--- |
| `s.value_counts()` | rows per distinct **value** of one column | `GROUP BY col` + `COUNT(*)` |
| `df.groupby(k).size()` | rows per **group** (includes NaN) | `GROUP BY k` + `COUNT(*)` |
| `s.nunique()` | number of **distinct** values | `COUNT(DISTINCT col)` |

**`drop_duplicates` vs `groupby().first()`:**

| | Keeps | Other columns |
| :--- | :--- | :--- |
| `df.drop_duplicates(subset=k)` | first row per key | kept as-is from that row |
| `df.groupby(k).first()` | first non-null per column per group | can differ column-by-column; lets you aggregate the rest |


---
## §7 — Window Operations

Covers ranking, lag/lead, running totals, and rolling windows — the Pandas equivalents of SQL window functions.

The important one is time-based rolling: `rolling(7)` means *seven rows*, `rolling('7D')` means *seven days* — they differ whenever the series has gaps.


In [ ]:
# Ranking — SQL RANK / DENSE_RANK / ROW_NUMBER equivalents
df['row_num']   = df['score'].rank(method='first',  ascending=False).astype(int)  # ROW_NUMBER
df['rnk']       = df['score'].rank(method='min',    ascending=False).astype(int)  # RANK
df['dense_rnk'] = df['score'].rank(method='dense',  ascending=False).astype(int)  # DENSE_RANK

# Ranking within a group — RANK() OVER (PARTITION BY ...)
df['rank_in_group'] = (
    df.groupby('region')['revenue']
      .rank(method='dense', ascending=False)
      .astype(int)
)

# LAG / LEAD — shift within a group (always sort first)
df = df.sort_values(['user_id', 'date'])
df['prev_value'] = df.groupby('user_id')['value'].shift(1)    # LAG(value, 1)
df['next_value'] = df.groupby('user_id')['value'].shift(-1)   # LEAD(value, 1)

# Running totals — SUM OVER (ROWS UNBOUNDED PRECEDING)
df['run_sum'] = df.groupby('user_id')['amount'].cumsum()
df['run_max'] = df.groupby('user_id')['amount'].cummax()
df['run_min'] = df.groupby('user_id')['amount'].cummin()

# Rolling window — moving average over last n rows
df['moving_avg_7'] = (
    df.groupby('user_id')['amount']
      .transform(lambda x: x.rolling(7, min_periods=1).mean())
)

# ── Row-based vs time-based windows ──────────────────────────────────────────
df['ma7_rows'] = df['sales'].rolling(7).mean()          # last 7 ROWS
df = df.set_index('date').sort_index()
df['ma7_days'] = df['sales'].rolling('7D').mean()       # last 7 DAYS (gap-aware)
# Time-based rolling REQUIRES a sorted DatetimeIndex.

# Centred window — aligns the label to the middle, not the right edge
df['smooth'] = df['sales'].rolling(7, center=True).mean()

# Exponentially weighted — recent points weighted more, no fixed cutoff
df['ewm'] = df['sales'].ewm(span=7).mean()              # span ~ "like a 7-period MA"
df['sales'].ewm(halflife='3D', times=df.index).mean()   # time-aware decay
df['sales'].ewm(alpha=0.3, adjust=False).mean()         # explicit smoothing factor

# Custom function over a window
df['rng'] = df['sales'].rolling(7).apply(lambda w: w.max() - w.min(), raw=True)
# raw=True passes a numpy array instead of a Series -> substantially faster

# Rolling relationships between two series
df['corr30'] = df['price'].rolling(30).corr(df['index_price'])
df['cov30']  = df['price'].rolling(30).cov(df['index_price'])
df['beta30'] = df['cov30'] / df['index_price'].rolling(30).var()

# Guard against thin windows at the start of the series
df['sales'].rolling(7, min_periods=7).mean()            # NaN until 7 obs exist

# Expanding window — cumulative from start of series
df['expanding_mean'] = df.groupby('user_id')['amount'].transform(lambda x: x.expanding().mean())


**Rank method comparison:**

| method | Ties | Gaps after ties | SQL equivalent |
| :--- | :--- | :--- | :--- |
| `first` | No — sequential | n/a | `ROW_NUMBER()` |
| `min` | Yes — share min rank | Yes | `RANK()` |
| `dense` | Yes — share rank | No | `DENSE_RANK()` |

**Window types:**

| | Window | Weights | Use when |
| :--- | :--- | :--- | :--- |
| `rolling(7)` | fixed 7 rows | equal | regular, gapless series |
| `rolling('7D')` | fixed 7 days | equal | irregular timestamps or missing days |
| `expanding()` | all prior rows | equal | cumulative-to-date metrics |
| `ewm(span=7)` | all prior rows | decaying | recent data matters more |

**Common mistakes:**
- Forgetting to `sort_values` before `shift` — shift operates on current row order, not logical order
- Using `groupby().cumsum()` without sorting — produces incorrect running totals
- Using `rolling()` directly on a grouped column without `transform` — loses the group alignment
- Using `rolling(7)` on a series with missing days and calling it a 7-day average — it is a 7-*observation* average
- Time-based rolling on an unsorted or non-datetime index — raises, or silently misaligns
- `center=True` in a production feature — it looks at future rows and leaks; only for visual smoothing
- Leaving `raw=False` in `rolling().apply()`, which builds a Series per window and is far slower
- Comparing `ewm(span=n)` to `rolling(n)` as if equivalent — `ewm` never fully forgets old data


---
## §8 — Date & Time

Always parse date strings with `pd.to_datetime()` first. All `.dt` accessor operations require a proper datetime dtype.

surviving unparseable dates, calendar-aware offsets, business days, and the boundary properties.


In [ ]:
# Parse strings to datetime
df['event_ts'] = pd.to_datetime(df['event_ts'])                     # auto-detect format
df['event_ts'] = pd.to_datetime(df['event_ts'], format='%Y-%m-%d')  # explicit format (faster)

# errors='coerce' — unparseable values become NaT instead of raising
df['date'] = pd.to_datetime(df['raw_date'], errors='coerce')
df['raw_date'][df['date'].isna()].unique()      # inspect what failed to parse

pd.to_datetime(df['d'], format='%d/%m/%Y', errors='coerce')   # explicit format: fastest + safest
pd.to_datetime(df['d'], format='mixed', dayfirst=True)        # genuinely mixed formats
pd.to_datetime(df['epoch'], unit='s')                         # unix seconds
pd.to_datetime(df['epoch_ms'], unit='ms')                     # unix milliseconds

# Unix timestamp conversion
df['dt'] = pd.to_datetime(df['ts_col'], unit='s')    # 10-digit (seconds) → datetime
df['dt'] = pd.to_datetime(df['ts_col'], unit='ms')   # 13-digit (milliseconds) → datetime

# Extract components — .dt accessor
df['date']        = df['event_ts'].dt.date                           # DATE()
df['year']        = df['event_ts'].dt.year                           # EXTRACT(YEAR)
df['month']       = df['event_ts'].dt.month                          # EXTRACT(MONTH)
df['day']         = df['event_ts'].dt.day
df['day_of_week'] = df['event_ts'].dt.dayofweek                      # 0=Mon, 6=Sun
df['hour']        = df['event_ts'].dt.hour
df['year_month']  = df['event_ts'].dt.to_period('M').astype(str)     # DATE_FORMAT('%Y-%m')

# ── Boundary properties ──────────────────────────────────────────────────────
d = df['date'].dt
d.is_month_end, d.is_month_start
d.is_quarter_end, d.is_year_end
d.days_in_month, d.dayofweek, d.dayofyear, d.isocalendar().week
df[d.dayofweek >= 5]                      # weekend rows

# Date arithmetic
df['plus_7d']   = df['order_date'] + pd.Timedelta(days=7)            # DATE_ADD(..., INTERVAL 7 DAY)
df['minus_1mo'] = df['order_date'] - pd.DateOffset(months=1)         # DATE_SUB(..., INTERVAL 1 MONTH)
df['days_diff'] = (df['end_date'] - df['start_date']).dt.days        # DATEDIFF

# ── Offsets: calendar-aware, unlike Timedelta ────────────────────────────────
from pandas.tseries.offsets import MonthEnd, MonthBegin, BDay, QuarterEnd, Week

df['date'] + MonthEnd(0)                  # snap to end of THIS month
df['date'] + MonthEnd(1)                  # end of next month
df['date'] - MonthBegin(1)                # start of this month
df['date'] + BDay(3)                      # 3 business days later (skips weekends)
df['date'] + QuarterEnd(0)                # end of current quarter
df['date'] + pd.DateOffset(months=1)      # calendar month: Jan 31 -> Feb 28/29

# Timedelta is a FIXED duration and does not know about calendars
df['date'] + pd.Timedelta(days=30)        # always exactly 30 days

# ── Business days ────────────────────────────────────────────────────────────
pd.bdate_range('2024-01-01', '2024-01-31')                # weekdays only
pd.bdate_range('2024-01-01', periods=10, freq='C',        # custom calendar
               holidays=['2024-01-15'])
np.busday_count('2024-01-01', '2024-02-01')               # count business days

# Current date
today = pd.Timestamp.today().normalize()                              # CURRENT_DATE
df_recent = df.loc[df['event_date'] >= today - pd.Timedelta(days=7)]

# Gaps & Islands — consecutive streak detection
logins = logins.sort_values(['user_id', 'login_date'])
logins['row_num'] = logins.groupby('user_id').cumcount() + 1
logins['grp'] = logins['login_date'] - pd.to_timedelta(logins['row_num'], unit='D')
streaks = (
    logins.groupby(['user_id', 'grp'], as_index=False)
          .agg(streak_start=('login_date', 'min'),
               streak_end  =('login_date', 'max'),
               streak_len  =('login_date', 'count'))
)


**`pd.Timedelta` vs `pd.DateOffset`:**

| | Fixed duration | Calendar-aware | Use for |
| :--- | :--- | :--- | :--- |
| `pd.Timedelta(days=7)` | ✅ | ❌ | Days, hours, minutes |
| `pd.DateOffset(months=1)` | ❌ | ✅ | Months, years (variable length) |

**`Timedelta` vs `DateOffset` vs offset objects:**

| | Jan 31 + 1 month | Skips weekends? |
| :--- | :--- | :--- |
| `Timedelta(days=30)` | Mar 1 | ❌ |
| `DateOffset(months=1)` | Feb 29 | ❌ |
| `MonthEnd(1)` | Feb 29 | ❌ |
| `BDay(1)` | next weekday | ✅ |

**Common mistakes:**
- Forgetting `pd.to_datetime()` before using `.dt` accessor — raises `AttributeError` if column is still a string
- Using `pd.Timedelta` for months — months have variable lengths; use `pd.DateOffset` instead
- Using `.dt.date` (returns Python `date` objects) when you need `.dt.normalize()` (returns `Timestamp`) — the former breaks further date arithmetic
- `errors='coerce'` without checking how many rows became `NaT` — silent data loss
- Letting `to_datetime` infer the format on a large column; it is slow and can flip day/month
- Using `Timedelta(days=30)` for "one month later" — wrong for every month that is not 30 days
- Assuming `MonthEnd(1)` from a date already at month end moves one month; use `MonthEnd(0)` to snap, `MonthEnd(1)` to advance


---
## §9 — String Operations

All string methods live under the `.str` accessor. They are vectorized and NaN-safe — NaN rows return NaN rather than raising an error.

ending with the one interviews actually probe: matching names that are *nearly* equal.


In [ ]:
# Case and whitespace
df['col'].str.lower()
df['col'].str.upper()
df['col'].str.strip()                           # remove leading/trailing whitespace
df['col'].str.strip().str.lower()               # chain operations

# Unicode normalisation — normalising case is not enough on its own
df['name'].str.normalize('NFKD')                                  # decompose accents
(df['name'].str.normalize('NFKD')
           .str.encode('ascii', 'ignore').str.decode('utf-8'))     # strip them entirely
df['name'].str.replace('\u00a0', ' ', regex=False)                 # non-breaking space

# Contains / starts with / ends with
df['col'].str.contains('pattern')               # LIKE '%pattern%'
df['col'].str.contains('pattern', na=False)     # treat NaN as False, not NaN
df['col'].str.startswith('prefix')
df['col'].str.endswith('suffix')

# Replace
df['col'].str.replace('old', 'new')             # REPLACE(col, 'old', 'new')
df['col'].str.replace(r'\d+', '', regex=True)   # remove all digits (regex)

# Split
df['col'].str.split(',')                        # returns list in each cell
df['col'].str.split(',', expand=True)           # expand into separate columns
df[['first', 'last']] = df['name'].str.split(' ', n=1, expand=True)  # split into named cols

# Extract — pull out groups using regex
df['col'].str.extract(r'(\d{4})')               # extract first 4-digit number into a column
df['col'].str.extract(r'(?P<year>\d{4})-(?P<month>\d{2})')  # named groups → named columns

# Find every match, not just the first
df['text'].str.findall(r'\d+')                    # list of all numbers per row
df['text'].str.count(r'\berror\b')                # occurrences per row
df['text'].str.extractall(r'(\d+)')               # long-format frame of all matches

# Length
df['col'].str.len()                             # character count per cell

# Slice
df['col'].str[:3]                               # first 3 characters
df['col'].str[2:5]                              # characters 2–4

# Padding and zero-fill (IDs, zip codes)
df['zip'].astype(str).str.zfill(5)                # '123' -> '00123'
df['code'].str.pad(10, side='right', fillchar='.')

# Concatenate across columns / down a Series
df['full'] = df['first'].str.cat(df['last'], sep=' ', na_rep='')
df['tags'].str.cat(sep=', ')                      # collapse a whole Series to one string

# One-hot encode a delimited column in a single call
df['tags'].str.get_dummies(sep='|')               # 'a|b' -> columns a, b

# ── Fuzzy matching — when keys are close but not equal ──────────────────────
import difflib
difflib.get_close_matches('Jonh Smith', known_names, n=3, cutoff=0.8)

df['match'] = df['raw_name'].apply(
    lambda x: (difflib.get_close_matches(x, known_names, n=1, cutoff=0.85) or [None])[0]
)
# For large joins prefer rapidfuzz (C++ backed, orders of magnitude faster):
#   from rapidfuzz import process
#   process.extractOne(x, known_names, score_cutoff=85)


**Extraction methods:**

| Method | Returns | Matches |
| :--- | :--- | :--- |
| `str.extract` | DataFrame, one row per input | first only |
| `str.extractall` | long DataFrame, MultiIndex | all |
| `str.findall` | Series of lists | all |
| `str.count` | Series of ints | all (count only) |

**Common mistakes:**
- `str.contains()` without `na=False` — NaN rows return NaN (not False), which breaks boolean indexing
- Using Python `str` methods directly on a Series instead of `.str.*` — raises `AttributeError`
- `str.replace()` without `regex=False` when the pattern has regex special characters (`.`, `+`, `*`) — always set `regex=False` for literal replacements
- Normalising case but not unicode — `"José"` and `"José"` can differ by encoding and never join
- `str.zfill` on a numeric column; cast to `str` first or it fails silently
- `str.get_dummies` on a high-cardinality column, which explodes the column count
- Fuzzy-matching in a loop over a large frame — `difflib` is O(n·m) and will hang; use `rapidfuzz`
- Forgetting `na=False` in `str.contains` when NaNs are present, which yields NaN and breaks the mask


---
## §10 — I/O

Covers reading and writing data. Setting dtypes on load avoids silent type errors downstream.

In [ ]:
# Read CSV
df = pd.read_csv('file.csv')
df = pd.read_csv('file.csv', usecols=['user_id', 'date', 'revenue'])  # load only needed columns
df = pd.read_csv('file.csv', dtype={'user_id': str, 'amount': float}) # specify dtypes on load
df = pd.read_csv('file.csv', parse_dates=['event_date'])              # parse dates on load
df = pd.read_csv('file.csv', nrows=1000)                              # load first N rows only

# Process large files in chunks
chunks = pd.read_csv('large.csv', chunksize=100_000)
result = pd.concat([chunk[chunk['revenue'] > 0] for chunk in chunks], ignore_index=True)

# Read JSON
df = pd.read_json('file.json')
df = pd.read_json('file.json', lines=True)                            # JSON Lines format

# Read Excel
df = pd.read_excel('file.xlsx', sheet_name='Sheet1')

# Read SQL
import sqlite3
conn = sqlite3.connect('db.sqlite')
df = pd.read_sql('SELECT * FROM table WHERE date > "2024-01-01"', conn)

# Write
df.to_csv('output.csv', index=False)            # index=False avoids writing the row index
df.to_json('output.json', orient='records')
df.to_excel('output.xlsx', index=False, sheet_name='Results')

**Common mistakes:**
- Forgetting `index=False` in `to_csv` — writes the numeric row index as the first column
- Loading all columns when only a few are needed — use `usecols` to reduce memory
- Not specifying `dtype` for ID columns — numeric IDs load as `int64` and lose leading zeros (e.g. zip codes)

---
## §11 — Performance Tips

These are the rules of thumb. For the same claims **measured** — timings for each rung of the iteration ladder, memory before and after a dtype pass, and the point where `category` stops paying off — see {doc}`Reference_Performance <Reference_Performance>`, which executes at build time.


In [ ]:
# Prefer vectorized operations over apply
df['col'] * 2                           # fast — vectorized C loop
df['col'].apply(lambda x: x * 2)        # slow — row-by-row Python loop

# .query() vs boolean indexing
df.query("age > 30 and region == 'US'") # readable, slightly faster on large DataFrames
df[(df['age'] > 30) & (df['region'] == 'US')]  # equivalent, better for dynamic conditions

# Reduce memory with category dtype (low-cardinality string columns)
df['platform'] = df['platform'].astype('category')

# Read only needed columns
pd.read_csv('file.csv', usecols=['user_id', 'date', 'revenue'])

# Avoid chained indexing — use .loc instead
df.loc[mask, 'col'] = value             # correct
df[mask]['col'] = value                 # wrong — may not modify original (SettingWithCopyWarning)

# Use .copy() when slicing a DataFrame you intend to modify
subset = df[df['region'] == 'US'].copy()
subset['new_col'] = 1                   # safe — modifies copy, not original

# Process large files in chunks
chunks = pd.read_csv('large.csv', chunksize=100_000)
result = pd.concat([process(chunk) for chunk in chunks])

- Avoid `apply` with `axis=1` (row-wise) — it is a Python loop; use `np.where`, `np.select`, or vectorized column ops instead
- `inplace=True` does not speed things up and prevents chaining — avoid it
- For string columns with few unique values (platform, country, status), `.astype('category')` can reduce memory by 50–90%
- `df.eval('a + b')` can be faster than `df['a'] + df['b']` for large DataFrames — uses `numexpr` under the hood

---
## Interview Q&A

**Q: You need every user paired with every day in January, including days with no activity. How?**
A: A cross join to build the scaffold, then a left join for the data.
`users[['user_id']].merge(pd.DataFrame({'date': pd.date_range(...)}), how='cross')`
gives the complete grid, and left-joining events onto it with a `fillna(0)` yields
zero rows rather than missing rows. Doing it the other way round — grouping the
events — silently drops days nobody was active, which is exactly the bug in most
retention calculations.

**Q: Join each trade to the prevailing quote at that moment.**
A: `merge_asof`, not `merge`. An equality join fails because timestamps rarely
match exactly. I'd sort both frames on time, use `by='symbol'` so matching stays
within an instrument, and set a `tolerance` so a stale quote from hours earlier
does not get attached silently.

**Q: `.loc` vs `.iloc` vs `.at`?**
A: `.loc` is label-based and its slices are end-inclusive; `.iloc` is positional
and end-exclusive; `.at` is a single-value fast path. The practical trap is that
`df.loc[0:5]` returns six rows while `df.iloc[0:5]` returns five.

**Q: An integer column turned into floats after a merge. Why?**
A: Unmatched rows introduced `np.nan`, which is a float, so the column widened to
`float64`. Using the nullable `Int64` dtype keeps the values integral and stores
missing as `pd.NA`.

**Q: Difference between `rolling(7)` and `rolling('7D')`?**
A: Seven rows versus seven calendar days. They agree only when the series has one
row per day with no gaps. On event data with missing days, `rolling(7)` quietly
spans a different amount of time for every row.

**Q: How do you check what changed between two versions of a table?**
A: `before.compare(after)` for cell-level differences on identically-labelled
frames, and `Index.difference` in both directions for rows added and removed. For
a quick summary I'd also merge with `indicator=True` and count the `_merge` column.

### Gotchas
- Chained indexing (`df['a'][mask] = x`) may write to a temporary copy; use `df.loc[mask, 'a'] = x`
- `merge` silently produces a row explosion on many-to-many keys — pass `validate='1:m'` to make it raise instead
- `how='cross'` has no key to deduplicate on, so it multiplies row counts; always check `len(left) * len(right)` first
- Comparing float columns with `==` after arithmetic; use `np.isclose`
- `inplace=True` is not faster and is being phased out — prefer reassignment

---
# Decision Guide

```
Aggregating data?
├── One metric, one grouping dimension        → groupby().agg()
├── One metric across two categorical dims    → pivot_table()
├── Frequency count only                      → crosstab()
└── Need result back on every original row    → groupby().transform()

Filtering rows?
├── Static, readable condition                → .query()
└── Dynamic / programmatic condition          → boolean indexing df[mask]

Selecting rows + columns together?
├── By label                                  → .loc[rows, cols]
└── By position                               → .iloc[rows, cols]

Labeling / bucketing values?
├── Binary condition                           → np.where(cond, true_val, false_val)
├── Multiple conditions                        → np.select(conditions, choices, default)
└── Binning continuous values                  → pd.cut() or pd.qcut()

Combining DataFrames?
├── Join on a column                           → .merge(on='col')
├── Rows with no match in other table          → merge(..., how='left', indicator=True)
└── Stack rows vertically                      → pd.concat([df1, df2])

Window operations?
├── Rank within group                          → groupby().rank(method='dense')
├── Previous / next row value                  → groupby().shift(1) / shift(-1)
├── Running total                              → groupby().cumsum()
└── Moving average over last n rows            → groupby().transform(lambda x: x.rolling(n).mean())

NULL handling?
├── Replace with a value                       → .fillna(value)
├── Replace with previous row in group         → groupby().ffill()
└── Drop rows                                  → .dropna(subset=[...])

Speeding up slow code?
├── Using apply row-wise                       → replace with np.where / vectorized ops
├── Large file loading slowly                  → usecols= or chunksize=
└── High-cardinality string columns            → .astype('category')
```